# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}\nVersion: {metadata.version}\n")

## 2. Data Overview
Review available record sets, their fields, and their `@id`s.

We'll enumerate the record sets, and for each record set, list field `@id`s and their names.

In [ ]:
# List available record sets and summarize their fields
record_sets = list(dataset.metadata.record_sets)
if not record_sets:
    print("No record sets defined in this dataset.")
else:
    for rs in record_sets:
        print(f"\nRecord set: {rs['@id']} (name: {rs.get('name', 'N/A')})")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        elif not isinstance(fields, list):
            fields = []
        print("Fields and their @id:")
        for field in fields:
            print(f"  {field['@id']}: {field.get('name', 'N/A')}")

    # If possible, show a sample record from the first record set
    example_rs_id = record_sets[0]['@id']
    print(f"\nSample record from record set {example_rs_id}:")
    for i, rec in enumerate(dataset.records(record_set=example_rs_id)):
        print(rec)
        if i > 0:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, using record set and field `@id`s as referenced above.

In [ ]:
# If you know the record set @id(s), set them here.
# If the previous cell found no record sets, try a fallback to an anticipated @id or use top-level records.
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets] if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets else []
if not record_set_ids:
    # Try to use the main record set id known for this dataset.
    # Here we must use the one defined in the schema if any, or manual inspection for the notebook to work.
    # You might replace the id below with the actual one after running the above cell.
    print("No record sets found. Please inspect the dataset schema for appropriate record set IDs.")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        print(f"Loading records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Choose the first available for further analysis
if dataframes:
    sample_record_set_id = list(dataframes.keys())[0]
    print(f"Proceeding with record set: {sample_record_set_id}")
else:
    sample_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. Use field `@id`s as shown earlier.

In [ ]:
if not sample_record_set_id:
    print("No record set available for EDA.")
    # You may wish to inspect the dataset metadata and update record_set_id and field IDs manually below.
else:
    df = dataframes[sample_record_set_id]
    # Try to pick a numeric field by inspecting dtypes
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]  # Use the first numeric field
        print(f"Numeric field for analysis: {numeric_field}")
        threshold = df[numeric_field].mean() if len(df) < 100 else df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Find a group field (categorical)
        group_fields = df.select_dtypes(include='object').columns.tolist()
        group_field = None
        if group_fields:
            # Heuristically use the first non-numeric, non-unique field
            for field in group_fields:
                if df[field].nunique() < len(df) / 2:
                    group_field = field
                    break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.reset_index().head())
        else:
            print("No suitable group field found for grouping analysis.")
    else:
        print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if sample_record_set_id and 'df' in locals() and not df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(data=df, x=numeric_field, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to load the FAIR^2 dataset on extracellular vesicle mass spectrometry using the Croissant metadata schema and the `mlcroissant` package.
- Fields, record sets, and their `@id`s were reviewed, and records were loaded programmatically.
- Basic exploratory steps and visualizations show how users can build custom analyses by referencing the schema's structure.

For in-depth field selection or further transformations, refer to the dataset metadata at the source Croissant JSON [here](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).
